## Massachusetts House Roll Call Parser
This notebook extracts and processes Massachusetts House of Representatives roll call voting records from PDF documents. It handles both multi-page PDFs (one roll call per page) and single-page PDFs (one roll call per PDF), supporting data from 2003-2025.

### 1. Import Dependencies
Load required libraries for PDF parsing, data manipulation, and file operations.

In [37]:
import re
import pdfplumber
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional

### 2. Metadata Extraction Functions

In [38]:
def extract_general_court_from_filename(filename: str) -> Optional[str]:
    """
    Extract the General Court Number from the file name
    """
    match = re.search(r'session[_\s]?(\d+)|^(\d{3})_', filename)
    if match:
        return match.group(1) or match.group(2)
    return None


def extract_title_text(text: str) -> str:
    """
    Extract the complete text of the page header row (the row containing the bill information)
    """
    lines = text.split('\n')
    title_lines = []
    found_header = False
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if 'MASSACHUSETTS HOUSE' in line.upper():
            found_header = True
            continue
        if re.search(r'\d+\s+YEAS?\s+\d+\s+NAYS?', line, re.IGNORECASE):
            break
        if found_header and line:
            if not re.match(r'(Yea and Nay|No\.\s*\d+)', line, re.IGNORECASE):
                title_lines.append(line)
    
    title_text = ' '.join(title_lines).strip()
    
    if not title_text:
        for line in lines:
            if re.match(r'[HS]\.\s*\d+|QUORUM|ELECTION|MOTION', line.strip(), re.IGNORECASE):
                title_text = line.strip()
                break
    
    return title_text


def extract_bill_number(title_text: str) -> Optional[str]:
    """
    Extract Bill Number from the title text
    """
    if not title_text:
        return None
    
    # Format 1: H. 57 or S. 123
    match = re.search(r'\b([HS])\.\s*(\d+)\b', title_text)
    if match:
        return f"{match.group(1)}. {match.group(2)}"
    
    # Format 2: H57 or S123
    match = re.search(r'\b([HS])(\d{3,5})\b', title_text)
    if match:
        return f"{match.group(1)}. {match.group(2)}"
    
    # Format 3: H 57 or S 123
    match = re.search(r'\b([HS])\s+(\d+)\b', title_text)
    if match:
        return f"{match.group(1)}. {match.group(2)}"
    
    # Format 4: (House, No. 3887)
    match = re.search(r'\((House|Senate),?\s*No\.\s*(\d+)', title_text, re.IGNORECASE)
    if match:
        chamber = 'H' if match.group(1).lower() == 'house' else 'S'
        return f"{chamber}. {match.group(2)}"
    
    return None


def extract_amendment_number(title_text: str) -> Optional[str]:
    """
    Extract the Amendment Number from the title text (must include the # symbol)
    """
    if not title_text:
        return None
    text = title_text.lower()
    # Model 1
    match = re.search(r'#\s*(\d+)', text)
    if match:
        return match.group(1)
    
    # Model 2: Amendment No. [Number]
    match = re.search(r'amend(?:ment|\.)\s*(?:no\.?)?\s*(\d+)', text)
    if match:
        return match.group(1)
    
    # Mode 3: Amendment Number 
    match = re.search(r'amend(?:ment|\.)\s*(\d+)', text)
    if match:
        return match.group(1)
    
    return None


def extract_rollcall_number(text: str) -> Optional[str]:
    """
    Extract Roll Call Number
    """
    match = re.search(r'(?:Yea and Nay|Roll Call)?\s*No\.\s*(\d+)', text, re.IGNORECASE)
    if match:
        return match.group(1)
    return None


def extract_date(text: str) -> Optional[str]:
    """
    Extract date
    """
    match = re.search(r'(\d{2}/\d{2}/\d{4}\s+\d{1,2}:\d{2}\s*[AP]M)', text)
    if match:
        return match.group(1).strip()
    return None

def extract_vote_counts(text: str):
    """
    Extract Yeas, Nays, NV counts from header text.
    """
    yeas = nays = nv = None

    # YEAS count
    m = re.search(r'(\d+)\s+YEAS?', text, re.IGNORECASE)
    if m:
        yeas = int(m.group(1))

    # NAYS count
    m = re.search(r'(\d+)\s+NAYS?', text, re.IGNORECASE)
    if m:
        nays = int(m.group(1))

    # N/V count 
    m = re.search(r'(\d+)\s+N[\/-]V', text, re.IGNORECASE)
    if m:
        nv = int(m.group(1))

    return yeas, nays, nv


### 3. Vote Record Extraction

In [ ]:
def extract_votes_from_page(page) -> List[Dict]:
    """
    Parse individual member votes from PDF page.
    """
    votes = []
    text = page.extract_text()
    
    if not text:
        return votes
    
    # Method 1: Table Extraction
    tables = page.extract_tables()
    if tables:
        for table in tables:
            for row in table:
                if not row:
                    continue
                
                # Handles 4-column layout: [Vote, Name, Vote, Name, Vote, Name, Vote, Name]
                for i in range(0, min(len(row), 8), 2):  # 最多8个元素（4组）
                    if i + 1 < len(row):
                        vote = str(row[i]).strip() if row[i] else ''
                        name = str(row[i + 1]).strip() if row[i + 1] else ''
                        
                        # Validates vote symbols
                        if vote in ['Y', 'N', 'X', '#', 'P'] and name and len(name) > 1:
                            votes.append({
                                'name': name,
                                'vote': vote
                            })
    
    # Method 2: Regex Fallback
    # Used when table extraction fails
    if not votes:
        for line in text.split('\n'):
            line = line.strip()
                
            if not line or 'member_name' in line.lower() or 'YEAS' in line:
                continue
            
            pattern = r'([YNXP#])\s+([\-A-Z][A-Za-z\.\-\'\s,]+?)(?=\s+[YNXP#]\s+|\s*$)'
            
            matches = re.finditer(pattern, line)
            
            for match in matches:
                vote = match.group(1).strip()
                name = match.group(2).strip()
                
                # Strips trailing punctuation from names
                name = re.sub(r'\s+', ' ', name)
                #name = name.rstrip('*-.,')
                
                if len(name) > 2:
                    votes.append({
                        'name': name,
                        'vote': vote
                    })
    
    seen = set()
    unique_votes = []
    for v in votes:
        key = v['name']
        if key not in seen:
            seen.add(key)
            unique_votes.append(v)
        
    
    return unique_votes

### 4. PDF Processing Functions

- Multi-Page PDF Format

In [ ]:
def parse_single_page(page, pdf_filename: str) -> Dict:
    """
    Process one page from a multi-page PDF (2018-2025 format).
    """
    text = page.extract_text()
    
    if not text:
        return None
    
    # Extract General Court Number
    general_court = extract_general_court_from_filename(pdf_filename)
    
    # Extract text from page
    rollcall_number = extract_rollcall_number(text)
    date = extract_date(text)
    title_text = extract_title_text(text)
    bill_number = extract_bill_number(title_text)
    amendment_number = extract_amendment_number(title_text)
    yeas_count, nays_count, nv_count = extract_vote_counts(text)
    # Parse all member votes
    votes = extract_votes_from_page(page)
    
    return {
        'GeneralCourtNumber': general_court,
        'RollcallNumber': rollcall_number,
        'Date': date,
        'TitleText': title_text,
        'BillNumber': bill_number,
        'AmendmentNumber': amendment_number,
        'YeasCount': yeas_count,
        'NaysCount': nays_count,
        'NVCount': nv_count,
        'Vote': votes
    }


def parse_rollcall_pdf(pdf_path: str) -> List[Dict]:
    """
    Analyze the entire PDF file 
    """
    results = []
    pdf_filename = Path(pdf_path).name
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                print(f"  Processing page {page_num}/{len(pdf.pages)}...", end='\r')
                
                page_data = parse_single_page(page, pdf_filename)
                
                if page_data and page_data['Vote']:
                    results.append(page_data)
        
        print(f"  ✓ Extracted {len(results)} roll calls from {len(pdf.pages)} pages")
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
    
    return results




- Single-Page PDF Format

In [ ]:
def parse_single_pdf(pdf_path: str) -> Optional[Dict]:
    """Process entire PDF as one roll call (2003-2017 format)."""
    pdf_filename = Path(pdf_path).name
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Merge text from all pages
            full_text = ''
            all_votes = []
            
            for page in pdf.pages:
                full_text += page.extract_text() + '\n'
                page_votes = extract_votes_from_page(page)
                all_votes.extend(page_votes)
            
            # Extract metadata from combined text
            general_court = extract_general_court_from_filename(pdf_filename)
            rollcall_number = extract_rollcall_number(full_text)
            date = extract_date(full_text)
            title_text = extract_title_text(full_text)
            bill_number = extract_bill_number(title_text)
            amendment_number = extract_amendment_number(title_text)
            yeas_count, nays_count, nv_count = extract_vote_counts(full_text)
            
            # Deduplicate member names
            seen = set()
            unique_votes = []
            for v in all_votes:
                if v['name'] not in seen:
                    seen.add(v['name'])
                    unique_votes.append(v)
            
            if unique_votes:
                return {  
                    'GeneralCourtNumber': general_court,
                    'RollcallNumber': rollcall_number,
                    'Date': date,
                    'TitleText': title_text,
                    'BillNumber': bill_number,
                    'AmendmentNumber': amendment_number,
                    'YeasCount': yeas_count,
                    'NaysCount': nays_count,
                    'NVCount': nv_count,
                    'Vote': unique_votes
                }
    
    except Exception as e:
        print(f"  ✗ Error: {e}")
    
    return None 

### 5. Batch Processing Functions

- Year-Based Processing

In [ ]:
def process_directory(directory: str, output_csv: str = 'rollcalls_data.csv', start_year=2018):
    """
    Process PDFs organized by year folders (2018+).
    """
    pdf_dir = Path(directory)
    all_records = []
    
    # Find all PDFs in year-named folders
    pdf_files = sorted(pdf_dir.rglob('*.pdf'))
    
    filtered_files = []
    
    for pdf_file in pdf_files:
        
        parent_folder = pdf_file.parent.name
        if parent_folder.isdigit():
            year = int(parent_folder)
            if year >= start_year:
                filtered_files.append(pdf_file)
        elif parent_folder.startswith('session_'):
            print(f"  Skipping (old format): {pdf_file.relative_to(pdf_dir)}")
            continue
    
    print(f"Found {len(pdf_files)} PDF files total")
    print(f"Processing {len(filtered_files)} files (>= {start_year})")
    print(f"Skipped {len(pdf_files) - len(filtered_files)} files (< {start_year} or old format)\n")
    
    for i, pdf_file in enumerate(filtered_files, 1):
        print(f"[{i}/{len(filtered_files)}] {pdf_file.name}")
        
        # Process each PDF page-by-page
        rollcalls = parse_rollcall_pdf(str(pdf_file))
        
        # Flatten vote records to CSV rows
        for rollcall in rollcalls:
            for vote in rollcall['Vote']:
                record = {
                    'GeneralCourtNumber': rollcall['GeneralCourtNumber'],
                    'RollcallNumber': rollcall['RollcallNumber'],
                    'Date': rollcall['Date'],
                    'TitleText': rollcall['TitleText'],
                    'BillNumber': rollcall['BillNumber'],
                    'AmendmentNumber': rollcall['AmendmentNumber'],
                    'YeasCount': rollcall.get('YeasCount'),
                    'NaysCount': rollcall.get('NaysCount'),
                    'NVCount': rollcall.get('NVCount'),
                    'RollCallName': vote['name'],
                    'Vote': vote['vote']
                }
                all_records.append(record)
    if all_records:
        df = pd.DataFrame(all_records)

        df['GeneralCourtNumber'] = df['GeneralCourtNumber'].fillna('').astype(str)
        df['RollcallNumber'] = df['RollcallNumber'].fillna('').astype(str)
        df['AmendmentNumber'] = df['AmendmentNumber'].fillna('').astype(str)
        df['BillNumber'] = df['BillNumber'].fillna('').astype(str)

        df.to_csv(output_csv, index=False, encoding='utf-8-sig')

        print(f"\n{'='*60}")
        print(f"✓ Saved {len(all_records)} records to {output_csv}")
        print(f"{'='*60}")

    # Summary
        print("\nSummary:")
        print(f"  Total Roll Calls: {df['RollcallNumber'].nunique()}")
        print(f"  Unique Members: {df['RollCallName'].nunique()}")
        print(f"  Date Range: {df['Date'].min()} to {df['Date'].max()}")

    # Content analysis
        has_bill = (df['BillNumber'] != '').sum()
        has_amend = (df['AmendmentNumber'] != '').sum()

        print("\nContent Analysis:")
        print(f"  Records with Bill Number: {has_bill} ({has_bill/len(df)*100:.1f}%)")
        print(f"  Records with Amendment: {has_amend} ({has_amend/len(df)*100:.1f}%)")
 
    else:
        print("\n✗ No data extracted!")


    
    return df if all_records else None


- Session-Based Processing

In [ ]:
def process_sessions(directory: str, sessions: List[int], output_csv: str = 'rollcalls_sessions.csv'):
    """
    Process PDFs organized by session folders (2003-2017).
    """
    pdf_dir = Path(directory)
    all_records = []
    
    # Find PDFs in session_XXX folders
    pdf_files = []
    for session in sessions:
        session_folder = pdf_dir / f"session_{session}"
        if session_folder.exists():
            pdf_files.extend(sorted(session_folder.glob('*.pdf')))
    
    print(f"Found {len(pdf_files)} PDF files from sessions {sessions}\n")
    
    for i, pdf_file in enumerate(pdf_files, 1):
        print(f"[{i}/{len(pdf_files)}] {pdf_file.name}", end='')
        
        rollcall_data = parse_single_pdf(str(pdf_file))
        
        if rollcall_data:
            print(f" ✓ ({len(rollcall_data['Vote'])} votes)")
        
            for vote in rollcall_data['Vote']:
                record = {
                    'GeneralCourtNumber': rollcall_data['GeneralCourtNumber'],
                    'RollcallNumber': rollcall_data['RollcallNumber'],
                    'Date': rollcall_data['Date'],
                    'TitleText': rollcall_data['TitleText'],
                    'BillNumber': rollcall_data['BillNumber'],
                    'AmendmentNumber': rollcall_data['AmendmentNumber'],
                    'YeasCount': rollcall_data.get('YeasCount'),
                    'NaysCount': rollcall_data.get('NaysCount'),
                    'NVCount': rollcall_data.get('NVCount'),
                    'RollCallName': vote['name'],
                    'Vote': vote['vote']
                }
                all_records.append(record)
        else:
            print(" ✗ No data")
    
    if all_records:
        df = pd.DataFrame(all_records)
        
        df['GeneralCourtNumber'] = df['GeneralCourtNumber'].fillna('').astype(str)
        df['RollcallNumber'] = df['RollcallNumber'].fillna('').astype(str)
        df['BillNumber'] = df['BillNumber'].fillna('').astype(str)
        df['AmendmentNumber'] = df['AmendmentNumber'].fillna('').astype(str)
        
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        
        print(f"\n{'='*70}")
        print(f"✓ Saved {len(all_records)} records to {output_csv}")
        print(f"{'='*70}")
        print(f"\nSummary:")
        print(f"  Total Roll Calls: {df['RollcallNumber'].nunique()}")
        print(f"  Unique Members: {df['RollCallName'].nunique()}")
        print(f"  Date Range: {df['Date'].min()} to {df['Date'].max()}")
        print(f"  Sessions: {sorted(df['GeneralCourtNumber'].unique())}\n")
        
        has_bill = (df['BillNumber'] != '').sum()
        has_amend = (df['AmendmentNumber'] != '').sum()
        print("Content Analysis:")
        print(f"  With Bill Number: {has_bill} ({has_bill/len(df)*100:.1f}%)")
        print(f"  With Amendment: {has_amend} ({has_amend/len(df)*100:.1f}%)")
        
        return df
    
    else:
        print("\n✗ No data extracted!")
        return None


In [ ]:
"""-----Test the extraction results for a single PDF file-----
def test_single_file(pdf_path: str):
    
    
    print(f"Testing: {pdf_path}\n")
    print("="*60)
    
    rollcalls = parse_rollcall_pdf(pdf_path)
    
    all_records = []
    for rollcall in rollcalls:
        for vote in rollcall['votes']:
            record = {
                'general_court_number': rollcall['general_court_number'],
                'rollcall_number': rollcall['rollcall_number'],
                'date': rollcall['date'],
                'title_text': rollcall['title_text'],
                'bill_number': rollcall['bill_number'],
                'amendment_number': rollcall['amendment_number'],
                'member_name': vote['name'],
                'vote': vote['vote']
            }
            all_records.append(record)
    if all_records:
        df = pd.DataFrame(all_records)
        
        print("="*80)
        print(f"Extracted {len(df)} records from {len(rollcalls)} roll call(s)")
        print("="*80)
        
        # 显示前20行
        print("\nFirst 20 records:")
        print(df.head(170).to_string(index=False))
        
        print(f"\n... Total {len(df)} records")
        
        # 统计信息
        print("\n" + "="*80)
        print("Summary:")
        print("="*80)
        print(f"  Roll Calls: {df['rollcall_number'].nunique()}")
        print(f"  Unique Members: {df['member_name'].nunique()}")
        print(f"  Bills: {df['bill_number'].dropna().nunique()}")
        print(f"  Amendments: {df['amendment_number'].dropna().nunique()}")
        
        return df
    else:
        print("No data extracted!")
        return None
"""

'-----Test the extraction results for a single PDF file-----\ndef test_single_file(pdf_path: str):\n\n\n    print(f"Testing: {pdf_path}\n")\n    print("="*60)\n\n    rollcalls = parse_rollcall_pdf(pdf_path)\n\n    all_records = []\n    for rollcall in rollcalls:\n        for vote in rollcall[\'votes\']:\n            record = {\n                \'general_court_number\': rollcall[\'general_court_number\'],\n                \'rollcall_number\': rollcall[\'rollcall_number\'],\n                \'date\': rollcall[\'date\'],\n                \'title_text\': rollcall[\'title_text\'],\n                \'bill_number\': rollcall[\'bill_number\'],\n                \'amendment_number\': rollcall[\'amendment_number\'],\n                \'member_name\': vote[\'name\'],\n                \'vote\': vote[\'vote\']\n            }\n            all_records.append(record)\n    if all_records:\n        df = pd.DataFrame(all_records)\n\n        print("="*80)\n        print(f"Extracted {len(df)} records from {l

### 6. Main Execution Block

In [41]:
if __name__ == "__main__":
    #test_single_file("./HJRollCalls/2022/192_2022_RollCalls.pdf")

    df = process_directory("./HJRollCalls", output_csv="rollcalls_2018plus.csv")
    
    # df = process_directory("./HJRollCalls/2022", output_csv="rollcalls_2022.csv")
    

  Skipping (old format): session_183/session183_rollcall_0001.pdf
  Skipping (old format): session_183/session183_rollcall_0002.pdf
  Skipping (old format): session_183/session183_rollcall_0003.pdf
  Skipping (old format): session_183/session183_rollcall_0004.pdf
  Skipping (old format): session_183/session183_rollcall_0005.pdf
  Skipping (old format): session_183/session183_rollcall_0006.pdf
  Skipping (old format): session_183/session183_rollcall_0007.pdf
  Skipping (old format): session_183/session183_rollcall_0008.pdf
  Skipping (old format): session_183/session183_rollcall_0009.pdf
  Skipping (old format): session_183/session183_rollcall_0010.pdf
  Skipping (old format): session_183/session183_rollcall_0011.pdf
  Skipping (old format): session_183/session183_rollcall_0012.pdf
  Skipping (old format): session_183/session183_rollcall_0013.pdf
  Skipping (old format): session_183/session183_rollcall_0014.pdf
  Skipping (old format): session_183/session183_rollcall_0015.pdf
  Skipping

In [ ]:
""" ---test single file---
def test_single_file(pdf_path: str):
    print(f"Testing: {pdf_path}\n")
    print("="*70)
    
    rollcall = parse_single_pdf(pdf_path)
    
    if rollcall:
        print(f"General Court: {rollcall['general_court_number']}")
        print(f"Roll Call No.: {rollcall['rollcall_number']}")
        print(f"Date: {rollcall['date']}")
        print(f"Title: {rollcall['title_text'][:150]}...")
        print(f"Bill: {rollcall['bill_number'] or '(none)'}")
        print(f"Amendment: {rollcall['amendment_number'] or '(none)'}")
        print(f"Votes: {len(rollcall['votes'])} members")
        
        print(f"\nSample votes:")
        for i, vote in enumerate(rollcall['votes'][:10], 1):
            print(f"  {i}. {vote['name']:30s} → {vote['vote']}")
    else:
        print("No data extracted!")
    
    print("\n" + "="*70)
    """


In [ ]:
if __name__ == "__main__":
    #test_single_file("./HJRollCalls/session_183/session183_rollcall_0001.pdf")
    
    # 示例2：处理session 183-185（2003-2010）
    df = process_sessions("./HJRollCalls", sessions=[183, 184, 185], output_csv="rollcalls_183_185.csv")
    
    # 示例3：只处理session 183（2003-2004）
    # df = process_sessions("./HJRollCalls", sessions=[183], output_csv="rollcalls_2003_2004.csv")
    

Found 1576 PDF files from sessions [183, 184, 185]

[1/1576] session183_rollcall_0001.pdf ✓ (159 votes)
[2/1576] session183_rollcall_0002.pdf ✓ (159 votes)
[3/1576] session183_rollcall_0003.pdf ✓ (159 votes)
[4/1576] session183_rollcall_0004.pdf ✓ (159 votes)
[5/1576] session183_rollcall_0005.pdf ✓ (159 votes)
[6/1576] session183_rollcall_0006.pdf ✓ (159 votes)
[7/1576] session183_rollcall_0007.pdf ✓ (159 votes)
[8/1576] session183_rollcall_0008.pdf ✓ (159 votes)
[9/1576] session183_rollcall_0009.pdf ✓ (158 votes)
[10/1576] session183_rollcall_0010.pdf ✓ (157 votes)
[11/1576] session183_rollcall_0011.pdf ✓ (153 votes)
[12/1576] session183_rollcall_0012.pdf ✓ (158 votes)
[13/1576] session183_rollcall_0013.pdf ✓ (158 votes)
[14/1576] session183_rollcall_0014.pdf ✓ (158 votes)
[15/1576] session183_rollcall_0015.pdf ✓ (158 votes)
[16/1576] session183_rollcall_0016.pdf ✓ (159 votes)
[17/1576] session183_rollcall_0017.pdf ✓ (158 votes)
[18/1576] session183_rollcall_0018.pdf ✓ (159 votes)
[19